## 第五章
* 本章主要围绕模型构建、参数访问和初始化、自定义层和块、模型读写、GPU加速等内容展开

### 5.1 层与块

- 很多时候设计架构的时候考虑的不是单层，而是更粗粒度的**块**，在网络设计的过程中"比单个层大"但“比整个模型小”的组件更有价值，"块"可以描述单个层、由多个层组成的组件或整个模型本身，通过定义代码来生成任意复杂度的块，可以通过简洁的代码实现复杂的神经网络。

- 从编程的角度来看，块需要由"类(class)"表示，类的任何子类都必须定义一个将其输入转换为输出的前向传播函数，并且必须存储任何必需的参数。最后，为了计算梯度，块必须具有反向传播函数。

- 下面回顾一下之前写的多层感知机的代码：

In [2]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20,256),
        nn.ReLU(),
        nn.Linear(256,10))

X = torch.rand(2,20) # [0,1)均匀分布的采样
net(X)

tensor([[ 0.0361,  0.0713,  0.0761, -0.2873, -0.1185,  0.0976, -0.2254, -0.0114,
         -0.0387, -0.1092],
        [ 0.0401,  0.0397,  0.0881, -0.2479, -0.0109,  0.0199, -0.2196,  0.0806,
         -0.0362, -0.0474]], grad_fn=<AddmmBackward0>)

- 上述的例子中，通过实例化nn.Sequential构建了网络模型，层内的执行顺序是作为参数传递的。nn.Sequential定义了一种特殊的Module，即在Pytorch中表示一个块的类，其中两个全连接层也是Linear的实例，Linear类本身也是Module的子类。
- 另外，使用net(X)调用模型来获得模型的输出，这实际上是net.__call__(X)的简写，这个前向传播的逻辑即将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。
- 根据我们前面的叙述，块需要完成一系列的基本功能：
    - 将输入数据作为其前向传播函数的参数
    - 通过前向传播函数来生成输出
    - 计算其输出关于输入的梯度，可通过其反向传播函数进行访问，这个过程通常是自动发生的
    - 存储和访问前向传播所需要的参数
    - 根据需要初始化模型参数
- 下面定义了一个MLP类继承表示块的类，实现只需要提供构造函数和前向传播函数

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)

    def forward(self,x):
        return self.out(F.relu(self.hidden(x)))

- 上述的前向传播函数中，以x作为输入，计算带有激活函数的隐藏表示，并输出未规范化的输出值。MLP这个实现中两个层都是实例变量。
- 除非需要实现一个新的运算符，否则都不需要担心反向传播函数或参数初始化，系统将自动生成这些组成。
- 最开始在定义两层的MLP时使用了nn.Sequential,Sequential类的设计是为了把其他模块串起来，如果希望构建一个自定义的MySequential，我们只需要实现两个关键函数：
    - 一个是将各个块追加到列表中的函数
    - 一个是前向传播函数，用于将输入按追加块的顺序传递给块组成的链条
- 这样的MySequential定义如下：

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module
    
    def forward(self,X):
        for block in self._modules.values():
            X = block(X)
        return X
    
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

# 这个实现中主要通过*args接收了多个层的设计，每个单独的block和layer顺序执行调用



tensor([[-0.0838,  0.0342, -0.0700, -0.1948,  0.0556,  0.0437,  0.2703, -0.1104,
          0.0591,  0.0681],
        [-0.1473,  0.0139, -0.1908, -0.2127,  0.0594,  0.0258,  0.3560, -0.1699,
          0.0439,  0.1870]], grad_fn=<AddmmBackward0>)

- 并不是所有的架构都是简单的顺序架构，回顾在前几章的内容中，Pytorch支持自定义复杂的控制流计算，因此在块的设计中我们也希望执行任意的数学运算
- 下面介绍一个随机运算的特例，即线性计算中有一部分参数是**常数参数**，这部分参数并不参与更新，也不是上一层的运算结果，例如：

In [7]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20,20), requires_grad=False)
        self.linear = nn.Linear(20,20)
    
    def forward(self,X):
        X = self.linear(X)
        X = F.relu(torch.mm(X,self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X

# 上面的模型定义中有几个需要注意的部分，
# 一个是+1的偏置，让更多的通过mm计算得到的结果经过relu能保持激活值
# 一个是self.linear的复用，forward中前后使用了两次self.linear，两次计算使用的是同一组参数，即两个全连接层共享参数
# 最后有一个是复杂的控制流调用，通过设置一定的终止条件，重复进行计算，即使是比较复杂的控制流，反向传播的自动计算模块也能很好的实现梯度捕捉

net = FixedHiddenMLP()
net(X)

tensor([[-0.0400, -0.0659, -0.0364,  0.0127, -0.0180, -0.0237, -0.0016, -0.0071,
          0.0161,  0.0052,  0.0038, -0.0235, -0.0160,  0.0256,  0.0415, -0.0152,
          0.0625,  0.0194,  0.0097,  0.0546],
        [-0.0389, -0.0499, -0.0283,  0.0138, -0.0114, -0.0113, -0.0076, -0.0028,
          0.0155, -0.0055, -0.0091, -0.0220, -0.0176,  0.0245,  0.0427, -0.0226,
          0.0539, -0.0027,  0.0174,  0.0316]], grad_fn=<DivBackward0>)

- 回顾Python语言的一些特性，类似C语言之类的纯编译型语言的标准执行流程是编译->汇编->链接->执行，即将编写的代码翻译成汇编语言，再有汇编语言编译成机器指令加上链接生成类似.exe文件之类的执行文件；但Python不同，现代的Python最常使用Cpython作为实现，CPython将编译和解释放在了一起，第一次运行代码的时候将Python代码编译为字节码，之后每次的运行都会逐行解释并执行
- 上述Python的执行方式导致Python有一个独特的性质，即全局解释器锁(Global Interpreter Lock),GIL这一机制限制了同一时刻只有一个线程能执行Python字节码，即使在多线程的环境下，GIL也会确保只有一个线程在运行Python代码，因此当出现计算密集型任务时，需要通过多进程、扩展为Numpy等方式绕过GIL，否则这一机制就会成为性能瓶颈
- 编写执行块的操作效率问题是编写块时需要注意的，回顾之前的代码，很多的自定义块都是在反复进行字典查找、代码执行和其他一些控制类的Python代码，这些就属于上述的计算密集型任务，因此后续需要围绕这一机制进行特定的优化

### 5.2 参数管理

- 超参数和架构选择完成之后，我们就进入训练阶段，训练阶段会根据预先制定的损失函数更新参数值，在训练后，更新后的参数需要参与未来的预测，有时这些参数十分有用，就比如一些场景中我们需要复用他们，将模型和参数保存下来，以便可以在其他软件中执行或者进行检查。
- 本小节的内容就包括：
    - 访问参数，用于调试、诊断和可视化
    - 参数初始化
    - 在不同模型组件间共享参数

#### 5.2.1 参数访问

In [12]:
import torch
import torch.nn as nn

net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,1))
X = torch.rand(size=(2,4))

# 可以通过索引来访问模型的任意层，模型本身就是一个列表
print(net[2].state_dict())
# 输出的结果显示这个线性层当中包含两类参数，即权重和偏置

# 如果要对参数执行任何操作，首先就需要能够访问底层的数值
print(type(net[2].bias)) 
# 输出结果为<class 'torch.nn.parameter.Parameter'>
# 根据上述的输出结果可以分析一下，nn.parameter是一个命名空间，nn.parameter.Parameter是上述命名空间当中的一个类，继承于torch.tensor，这个类当中的实例会被自动注册为模型的参数，并参与梯度计算和优化
# nn.Module.parameters()是一个方法，用于返回模型中所有的Parameter对象，是一个return的迭代器
# 我们需要nn.Parameter这种类的原因是使得nn.Module能够自动管理模型参数，nn.Parameter会自动注册为模型参数

# 因此我们需要更进一步提取这个类当中真正封装的数值，即：
print(net[2].bias.data)
print(net[2].weight.grad == True) # nn.Parameter会默认分配梯度

# 如果需要处理一个复杂的块，逐个访问参数会变得很麻烦，因此可以使用一些递归的方式来提取每个子块的参数，需要学习一下下面的这种写法：
print(*[(name, param.shape) for name,param in net[0].named_parameters()])

# 根据上面的name输出结果，因此在访问子块的参数时还可以像下面一样写:
print(net.state_dict()['2.bias'].data)

OrderedDict([('weight', tensor([[ 0.1778,  0.0604, -0.2300, -0.2706,  0.3428,  0.2420,  0.3445,  0.1305]])), ('bias', tensor([-0.1285]))])
<class 'torch.nn.parameter.Parameter'>
tensor([-0.1285])
False
('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
tensor([-0.1285])


In [13]:
# 如果整个网络由嵌套块构成时：
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),
                         nn.Linear(8,4),nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return net
# 注意add_module和需要命名的写法

rgnet = nn.Sequential(block2(), nn.Linear(4,1))

print(rgnet)
# 这样会输出整个网络的结构

# 也可以根据嵌套的索引来读取参数数值
print(rgnet[0][1][0].weight.data)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)
tensor([[-0.1002, -0.3481,  0.3851,  0.0432],
        [-0.1238,  0.1116, -0.1565, -0.3211],
        [-0.0143, -0.4588,  0.1640,  0.1610],
        [-0.3073,

#### 5.2.2 参数初始化